Dataset choice and pre-processing data 

In [3]:
# Import the dataset
import pandas as pd

df = pd.read_csv("online_shoppers_intention.csv")

print("Column names:", df.columns.tolist())
print("Dataset shape:", df.shape)

# Data preprocessing:
# One-hot encode categorical variables
categorical_cols = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

# Keep features and target separate after encoding
X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)

print("Shape after one-hot encoding:", df.shape)

# Split the dataset into training and testing sets
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set: {X_test.shape[0]} rows")

Column names: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']
Dataset shape: (12330, 18)
Shape after one-hot encoding: (12330, 75)
Training set: 9864 rows
Testing set: 2466 rows


Logistic Regression

In [4]:
from sklearn.linear_model import LogisticRegression as LRClassifier
from sklearn.pipeline import make_pipeline as build_lr_pipeline
from sklearn.preprocessing import StandardScaler as LRScaler
from sklearn.metrics import (
    accuracy_score as lr_accuracy_metric,
    roc_auc_score as lr_auc_metric,
    precision_score as lr_precision_metric,
    recall_score as lr_recall_metric,
    f1_score as lr_f1_metric,
    matthews_corrcoef as lr_mcc_metric,
)
import joblib

# Copy the data for Logistic Regression
lr_training_matrix = X_train.copy()
lr_testing_matrix = X_test.copy()
lr_training_labels = y_train.astype(int).copy()
lr_testing_labels = y_test.astype(int).copy()

# Build and train a scaled Logistic Regression pipeline
lr_fitted_pipeline = build_lr_pipeline(
    LRScaler(),
    LRClassifier(max_iter=1000, random_state=42),
)
lr_fitted_pipeline.fit(lr_training_matrix, lr_training_labels)

# Generate test predictions and positive-class probabilities
lr_test_predictions = lr_fitted_pipeline.predict(lr_testing_matrix)
lr_test_probabilities = lr_fitted_pipeline.predict_proba(lr_testing_matrix)[:, 1]

# Evaluate the trained model
lr_metric_results = {
    "Accuracy": lr_accuracy_metric(lr_testing_labels, lr_test_predictions),
    "AUC Score": lr_auc_metric(lr_testing_labels, lr_test_probabilities),
    "Precision": lr_precision_metric(
        lr_testing_labels, lr_test_predictions, zero_division=0
    ),
    "Recall": lr_recall_metric(
        lr_testing_labels, lr_test_predictions, zero_division=0
    ),
    "F1 Score": lr_f1_metric(
        lr_testing_labels, lr_test_predictions, zero_division=0
    ),
    "Matthews Correlation Coefficient": lr_mcc_metric(
        lr_testing_labels, lr_test_predictions
    ),
}

for lr_metric_name, lr_metric_value in lr_metric_results.items():
    print(f"{lr_metric_name}: {lr_metric_value:.4f}")

# Save the fitted pipeline, including its scaler, for later use
lr_model_path = "logistic_regression_metrics_pipeline.joblib"
joblib.dump(lr_fitted_pipeline, lr_model_path)
print(f"Saved trained model to: {lr_model_path}")

Accuracy: 0.8816
AUC Score: 0.8809
Precision: 0.7394
Recall: 0.3639
F1 Score: 0.4877
Matthews Correlation Coefficient: 0.4641
Saved trained model to: logistic_regression_metrics_pipeline.joblib
